# Text Generation

### Greedy Search Decoding

In [1]:
import torch 
from transformers import AutoTokenizer , AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "gpt2-xl"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

2025-09-19 16:25:09.940371: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758299110.133711      60 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758299110.186596      60 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

*Generate the next word*

In [2]:
import pandas as pd 
input_txt = "Transformers are the"

input_ids = tokenizer(input_txt,return_tensors="pt")["input_ids"].to(device)
iterations=[]
n_steps = 8
choices_per_step =10

with torch.no_grad():
    for _ in range(n_steps):
        iteration=dict()
        iteration["Input"]=tokenizer.decode(input_ids[0])
        output = model(input_ids=input_ids)
        next_token_logits = output.logits[0,-1,:]
        next_token_probs = torch.softmax(next_token_logits,dim=-1)
        sorted_ids=torch.argsort(next_token_probs,dim=-1,descending=True)
        for choice_idx in range(choices_per_step):
            token_id = sorted_ids[choice_idx]
            token_prob = next_token_probs[token_id].cpu().numpy()
            token_choice=(
                f"{tokenizer.decode(token_id)} ({100*token_prob:.2f}%)"
            )
            iteration[f"Choice{choice_idx+1}"] =token_choice
        input_ids = torch.cat([input_ids,sorted_ids[None,0,None]],dim=-1)
        iterations.append(iteration)
pd.DataFrame(iterations)

,Input,Choice1,Choice2,Choice3,Choice4,Choice5,Choice6,Choice7,Choice8,Choice9,Choice10
0,Transformers are the,most (8.53%),only (4.96%),best (4.65%),Transformers (4.37%),ultimate (2.16%),perfect (2.02%),first (1.86%),biggest (1.23%),last (1.00%),greatest (0.99%)
1,Transformers are the most,popular (16.78%),powerful (5.37%),common (4.96%),famous (3.72%),successful (3.20%),dangerous (1.92%),important (1.78%),iconic (1.33%),likely (1.25%),well (1.19%)
2,Transformers are the most popular,toy (10.63%),toys (7.23%),Transformers (6.60%),of (5.46%),and (3.76%),robot (2.23%),", (2.12%)",robots (1.87%),cartoon (1.48%),theme (1.32%)
3,Transformers are the most popular toy,line (34.38%),in (18.20%),of (11.71%),brand (6.10%),line (2.69%),lines (2.11%),on (1.84%),- (1.36%),series (1.25%),franchise (1.21%)
4,Transformers are the most popular toy line,in (46.28%),of (15.09%),", (4.94%)",on (4.40%),ever (2.72%),for (2.34%),. (1.99%),to (1.93%),among (1.76%),from (1.63%)
5,Transformers are the most popular toy line in,the (65.99%),history (12.42%),America (6.91%),Japan (2.44%),North (1.40%),all (0.71%),China (0.62%),Europe (0.55%),our (0.35%),existence (0.31%)
6,Transformers are the most popular toy line in the,world (69.26%),United (4.55%),history (4.29%),US (4.23%),U (2.30%),universe (1.89%),country (1.76%),UK (0.93%),entire (0.80%),toy (0.67%)
7,Transformers are the most popular toy line in ...,", (39.73%)",. (30.64%),and (9.87%),with (2.32%),today (1.74%),right (1.22%),( (1.19%),! (1.04%),; (0.99%),for (0.69%)


In [3]:
input_ids = tokenizer(input_txt,return_tensors="pt")["input_ids"].to(device)
output = model.generate(input_ids,max_new_tokens=n_steps,do_sample=False)
print(tokenizer.decode(output[0]))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Transformers are the most popular toy line in the world,


In [4]:
max_length = 1000
input_txt = """In a shocking finding, scientist discovered \
a herd of unicorns living in a remote, previously unexplored \
valley, in the Andes Mountains. Even more surprising to the \
researchers was the fact that the unicorns spoke perfect English.\n\n
"""
input_ids = tokenizer(input_txt,return_tensors="pt")["input_ids"].to(device)
output_greedy = model.generate(input_ids,max_length=max_length,do_sample=False)
print(tokenizer.decode(output_greedy[0]))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In a shocking finding, scientist discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English.


The researchers, from the University of California, Davis, and the University of Colorado, Boulder, were conducting a study on the Andean cloud forest, which is home to the rare species of cloud forest trees.


The researchers were surprised to find that the unicorns were able to communicate with each other, and even with humans.


The researchers were surprised to find that the unicorns were able to communicate with each other, and even with humans.

The researchers were surprised to find that the unicorns were able to communicate with each other, and even with humans.

The researchers were surprised to find that the unicorns were able to communicate with each other, and even with humans.

The researchers were surprised to find that the unicorns were able 

### Beam Search Decoding

In [5]:
import numpy as np
sum([np.log(0.5)]*1024)


-709.7827128933695

In [6]:
import torch.nn.functional as F
def log_probs_from_logits(logits,labels):
    logp = F.log_softmax(logits,dim=-1)
    logp_label = torch.gather(logp,2,labels.unsqueeze(2)).squeeze(-1)
    return logp_label

In [7]:
def sequence_logprob(model,labels,input_len=0):
    with torch.no_grad():
        output= model(labels)
        log_probs = log_probs_from_logits(
            output.logits[:,:-1,:],labels[:,1:]
        )
        seq_log_prob  =torch.sum(log_probs[:,input_len:])
    return seq_log_prob.cpu().numpy()

In [8]:
logp = sequence_logprob(model,output_greedy,input_len=len(input_ids[0]))
print(tokenizer.decode(output_greedy[0]))
print(f"\nlog-prob:{logp:.2f}")

In a shocking finding, scientist discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English.


The researchers, from the University of California, Davis, and the University of Colorado, Boulder, were conducting a study on the Andean cloud forest, which is home to the rare species of cloud forest trees.


The researchers were surprised to find that the unicorns were able to communicate with each other, and even with humans.


The researchers were surprised to find that the unicorns were able to communicate with each other, and even with humans.

The researchers were surprised to find that the unicorns were able to communicate with each other, and even with humans.

The researchers were surprised to find that the unicorns were able to communicate with each other, and even with humans.

The researchers were surprised to find that the unicorns were able 

In [9]:
output_beam = model.generate(input_ids,max_length=max_length,num_beams =5,do_sample=False)
logp = sequence_logprob(model,output_beam,input_len=len(input_ids[0]))
print(tokenizer.decode(output_beam[0]))
print(f"\nlog-prob:{logp:.2f}")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In a shocking finding, scientist discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English.


The discovery of the unicorns was made by a team of scientists from the University of California, Santa Cruz, and the National Geographic Society.


The scientists were conducting a study of the Andes Mountains when they discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains.


The scientists were conducting a study of the Andes Mountains when they discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. The scientists were conducting a study of the Andes Mountains when they discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. The scientists were conducting a study of the Andes Mountains when they discovered a her

In [10]:
output_beam = model.generate(input_ids,max_length=max_length,num_beams=5,do_sample=False,no_repeat_ngram_size=2)
logp = sequence_logprob(model,output_beam,input_len=len(input_ids[0]))
print(tokenizer.decode(output_beam[0]))
print(f"\nlog-prob:{logp:.2f}")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In a shocking finding, scientist discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English.


The discovery was made by a team of scientists from the University of California, Santa Cruz, and the National Geographic Society.

According to a press release, the scientists were conducting a survey of the area when they came across the herd. They were surprised to find that they were able to converse with the animals in English, even though they had never seen a unicorn before. The researchers believe that this is the first documented case of an animal communicating with humans in this way.<|endoftext|>

log-prob:-113.14


### Sampling Methods 

In [11]:
output_temp = model.generate(input_ids,max_length=max_length,do_sample=True,temperature=2.0,top_k=0)
print(tokenizer.decode(output_temp[0]))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In a shocking finding, scientist discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English.


Led multico Department Nevertheless maintaining Path beer still doesn translated strategies and causes deadly Class Controller Centauri friendHathave group rejoledge Rouosaurus irrewarEy procenturion bullshit Nazi degreeschid bee conventionanti pancakes contraceptionhouseStaff Grants by Folk futile draconian frequenciesud quasi Chevy bound blumpsonic question degrading wicked withholdingixelamas wayNew States warehouseBlack sci Fritz chronicitting chewingvine twice praise noServerughtices allegingIDszeb craftedHaving fungus 28 inaccurate soboli626Thanksveland take👤 a 🐂 Bender exclusion + snipersHappy success sumLeandem scentCI corruption clitor Patty God livingetsk075286Blue lamamera CelebrationYet correct Daw Motorsport adjust veget capacityPhill Modern so

In [12]:
output_temp = model.generate(input_ids,max_length=max_length,do_sample=True,
                             temperature=0.5,top_k=0)
print(tokenizer.decode(output_temp[0]))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In a shocking finding, scientist discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English.


The valley was located in the San Juan Mountains, in the Andes Mountains of northern Argentina, and was a previously unexplored area. The researchers were able to witness the unicorn herd from a helicopter and were able to capture footage of the herd and their behavior. They were able to observe the herd for over a month, and even had the opportunity to photograph the herd.


The researchers were able to observe the herd for over a month, and even had the opportunity to photograph the herd.

The researchers were able to observe the herd for over a month, and even had the opportunity to photograph the herd. During this time, the researchers observed the unicorn herd interact with each other, as well as with humans. The researchers also observed the herd inte

### Top-k and Nucleus(Top-p) Sampling Method

In [ ]:
output_topk = model.generate(input_ids,max_length=max_length,do_sample=True,
                             top_k=50)
print(tokenizer.decode(output_topk[0]))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In a shocking finding, scientist discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English.


"The idea that we can go there in any way that hasn't quite been tried before, I think is exciting. It's a way of investigating the language continuum of animals and what's unique about them. I guess one of the concerns for the researchers was about the perception by the public of the creatures themselves. Are they really real, or were they manufactured in some way?" says Dr Brian Hare, co-leader of the study conducted by the University of Otago.


With the help of GPS mapping software and remote camera traps, the researchers captured video of the unicorns in their natural habitat.


In the video, the researchers noticed the animals were wearing reflective, camouflage clothing, with a different shade for blue. This is evidence that the creatures are able to

In [16]:
output_topp = model.generate(input_ids,max_length=max_length,do_sample=True,
                             top_p=0.90)
print(tokenizer.decode(output_topp[0]))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In a shocking finding, scientist discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English.


The scientists studied the animals in order to determine the extent of their range, which has not been fully explored. This is one of the biggest finds ever made in the Andes Mountains. According to their research, the valley is located at an elevation of 4,500 meters above sea level, which is a little more than 2,000 meters above the nearest point of the Arctic Circle.

"This is the first time that unicorns have been recorded outside of a zoo and this may be the only example in the world outside of North America," Dr. Jorge Chabat said.


In order to observe the animals, the scientists enlisted the help of their friend and fellow scientist, Marco Sartorio, who also holds an M.Sc. degree in Ecology and Evolution. The animals were tranquilized and then moved